> **Notebook 1 de 3 de la parte de ESPRESSO** (LHS 1140, COSMIA UdeA).
> Orden: **01 extracción** (este) → **02 descarga** → LBL (`lbl/wrap_LHS1140.py`) → RadVel → **03 tránsitos y masas**.
>
> Este cuaderno ya estaba comentado, así que aquí no se cambió nada. Es el que explica paso a paso cómo se habla con el archivo de ESO; el 02 usa todo esto para bajar de verdad los espectros que van a LBL.

# Extracción de datos de LHS 1140 (TOI-256) desde el archivo de ESO

**Semillero COSMIA, UdeA.** Notebook de JuanDa, comentado para el equipo.

## Para qué sirve este notebook

Es el primer paso del proyecto: **encontrar y descargar** los espectros de LHS 1140 que están en el archivo público de ESO, para después sacarles las velocidades radiales con **LBL** (Line-By-Line, Artigau et al. 2022).

Lo que se hace, en orden:

1. Pedir las coordenadas de la estrella por su nombre.
2. Conectarse al archivo de ESO.
3. Buscar todas las observaciones que hay cerca de la estrella y contarlas por instrumento.
4. Separar HARPS y ESPRESSO, y partir ESPRESSO por fecha.
5. Mirar qué archivos componen un "producto" (una observación).
6. Descargar y abrir un archivo para revisarlo.
7. **(Nuevo, para HARPS)** Abrir el `.tar` de un producto de HARPS, encontrar el espectro por órdenes (`e2ds`) y ver qué blaze usó.

**División del trabajo:** HARPS lo continúa el compañero; ESPRESSO lo continúa JuanDa. Las secciones 1 a 6 son comunes y la sección 7 es el punto de partida para HARPS.

## Tres ideas antes de empezar

- **Objeto, producto y archivo no son lo mismo.** El *objeto* es la estrella. Un *producto* (`dp_id`, del tipo `ADP.2015-...`) es **una observación** ya procesada por el pipeline de ESO. Cada producto es un **paquete con varios archivos**: el espectro 1D principal, vistas previas, y un `.tar` con los productos intermedios.
- **LBL no usa el espectro 1D principal.** Necesita el espectro **separado por órdenes** del espectrógrafo: `e2ds` en HARPS, `S2D` en ESPRESSO. Ese viene dentro del `.tar`.
- **Las secciones 1 a 5 no descargan nada.** Solo le hacen preguntas al **catálogo** del archivo, es decir, a los metadatos.

Este notebook se escribió y corrió en Google Colab (Python 3.13).

## 0. Instalación

`pyvo` no viene en Colab, así que hay que instalarlo. `requests`, `astropy`, `numpy` y `matplotlib` ya vienen.

In [ ]:
%pip install pyvo

## Librerías

Qué es cada una y para qué la usamos:

| Librería | Para qué |
|---|---|
| `astropy.coordinates.SkyCoord` | Manejar posiciones en el cielo; pedir coordenadas por nombre. |
| `astropy.time.Time` | Convertir fechas entre formatos (MJD ↔ fecha normal). |
| `astropy.io.fits` | Abrir archivos FITS, el formato estándar de la astronomía. |
| `astropy.table.Table` | Tablas con columnas con nombre (como una hoja de cálculo). |
| `pyvo` | El "mensajero" para hacer preguntas a archivos astronómicos (protocolo TAP). |
| `numpy` | Arreglos y conteos. |
| `requests` | Descargar archivos de internet. |
| `pathlib.Path` | Manejar rutas de carpetas y archivos. |
| `matplotlib` | Graficar. |
| `tarfile` | Abrir archivos `.tar`, que son "cajas" con varios archivos adentro (sección 7). |

In [ ]:
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.io import fits
from astropy.table import Table
import pyvo
import numpy as np
import requests
from pathlib import Path
import matplotlib.pyplot as plt
import tarfile
import io

## 1. Coordenadas de la estrella

`SkyCoord.from_name()` le pregunta por internet a **Sesame** (el servicio de nombres del CDS, el mismo de SIMBAD) dónde está el objeto. Se usa en vez de escribir las coordenadas a mano para evitar errores de tipeo.

In [ ]:
LHS_1140_coord = SkyCoord.from_name('LHS 1140')
print(LHS_1140_coord)

El archivo de ESO pide las coordenadas en **grados**, como números simples. `.ra.deg` y `.dec.deg` hacen esa conversión.

In [ ]:
# Coordenadas en grados (números simples, que es lo que pide ADQL)
ra = LHS_1140_coord.ra.deg
dec = LHS_1140_coord.dec.deg
print(f'RA (en grados): {ra}')
print(f'Dec (en grados): {dec}')

## 2. Conectarse al archivo de ESO

**Cómo funciona:** el archivo de ESO es como una biblioteca con un **catálogo** (una tabla gigante donde cada fila describe una observación). **TAP** es la "ventanilla" estándar para hacerle preguntas a ese catálogo, y **ADQL** es el idioma de las preguntas (casi igual a SQL). `pyvo` lleva la pregunta y trae la respuesta.

**¿De dónde sale esta URL?** De la página de acceso programático de ESO: https://archive.eso.org/programmatic. Ahí se explica que el archivo tiene un servicio TAP en `https://archive.eso.org/tap_obs` y que se puede usar con `pyvo`.

Crear el objeto `tap` **todavía no pregunta nada**: solo representa la ventanilla.

In [ ]:
tap = pyvo.dal.TAPService('https://archive.eso.org/tap_obs')
print(tap)

## 3. ¿Qué información tiene el catálogo?

Antes de preguntar algo hay que saber qué columnas existen. La tabla principal se llama `ivoa.ObsCore` (es un estándar internacional, por eso los nombres de columna son iguales en muchos archivos). Traemos **1 sola fila** con **todas** las columnas (`SELECT TOP 1 *`) para ver sus nombres.

In [ ]:
pregunta = 'SELECT TOP 1 * FROM ivoa.ObsCore'
respuesta = tap.search(pregunta)       # aquí sí se manda la pregunta
fila = respuesta.to_table()            # la respuesta convertida en tabla de astropy

print(f'{len(fila.colnames)} columnas')
print(fila.colnames)

Las columnas que usamos y su significado:

| Columna | Qué guarda |
|---|---|
| `dp_id` | Identificador del producto (`ADP.…`). |
| `instrument_name` | Instrumento (`HARPS`, `ESPRESSO`, …). |
| `target_name` | Nombre del objeto según el observador (no siempre consistente). |
| `s_ra`, `s_dec` | Posición, en grados. |
| `t_min` | Inicio de la observación, en **MJD**. |
| `dataproduct_type` | Tipo de dato (`spectrum`, `image`, …). |
| `snr` | Señal a ruido. |
| `access_url` | Enlace al servicio DataLink del producto. |

**Para ver la descripción oficial y las unidades de todas las columnas** se puede consultar la tabla especial `TAP_SCHEMA.columns`:
```sql
SELECT column_name, description, unit FROM TAP_SCHEMA.columns WHERE table_name = 'ivoa.ObsCore'
```

## 4. ¿Qué observaciones hay de LHS 1140?

Se busca **por posición** y no por nombre, porque cada observador escribe el nombre a su manera (`LHS1140`, `LHS 1140`, `TOI-256`…).

**Cómo se lee la pregunta:**
- `SELECT …`: las columnas que queremos.
- `FROM ivoa.ObsCore`: la tabla (el catálogo).
- `WHERE CONTAINS(POINT(...), CIRCLE(...)) = 1`: solo las observaciones cuyo punto en el cielo cae dentro de un círculo alrededor de la estrella.
  - `POINT('ICRS', s_ra, s_dec)`: la posición de **cada observación**. Son nombres de columnas, por eso van sin `{}`; el servidor los lee fila por fila.
  - `CIRCLE('ICRS', {ra}, {dec}, {radio})`: el círculo centrado en la estrella. Van con `{}` porque son **variables de Python** que el f-string pega en el texto.
  - `'ICRS'`: el sistema de coordenadas estándar.
- El radio va en **grados**: 1/60 = 1 minuto de arco.

**Ojo:** Python no revisa el ADQL. Para Python es solo un texto, así que los errores de escritura en la pregunta (por ejemplo un `.` en vez de una `,`) recién aparecen cuando se manda con `tap.search()`, como un `DALQueryError`.

In [ ]:
radio = 1/60   # 1 arcmin, en grados

pregunta = f'''
SELECT dp_id, instrument_name, target_name, t_min, dataproduct_type, snr,
access_url
FROM ivoa.ObsCore
WHERE CONTAINS(POINT('ICRS', s_ra, s_dec),
               CIRCLE('ICRS', {ra}, {dec}, {radio})) = 1
'''

print(pregunta)   # revisar a ojo el texto final, con los números ya pegados

In [ ]:
obs = tap.search(pregunta).to_table()
print(f'Productos encontrados: {len(obs)}')
print(obs)

### Número de productos por instrumento

Cada fila de `ivoa.ObsCore` es un **producto** (una observación), no un archivo.

**Forma 1, en Python:** `np.unique(..., return_counts=True)` da los valores distintos de una columna y cuántas veces aparece cada uno.

In [ ]:
instrumentos, valores = np.unique(obs['instrument_name'], return_counts=True)
for inst, n in zip(instrumentos, valores):
    print(f'{inst:10s} {n}')

**Forma 2, que cuente el servidor:** `GROUP BY` agrupa las filas por instrumento y `COUNT(*)` cuenta cuántas hay en cada grupo. Además se filtra por `dataproduct_type = 'spectrum'` para quedarnos solo con espectros.

**Resultado obtenido:** HARPS 296, ESPRESSO 117 y CRIRES 1. Coincide con lo que reportan Cadieux et al. (2024) para ESPRESSO (117 espectros).

In [ ]:
pregunta_conteo = f'''
SELECT instrument_name, dataproduct_type, COUNT(*) AS n
FROM ivoa.ObsCore
WHERE CONTAINS(POINT('ICRS', s_ra, s_dec),
               CIRCLE('ICRS', {ra}, {dec}, {radio})) = 1 AND dataproduct_type = 'spectrum'
GROUP BY instrument_name, dataproduct_type
ORDER BY n DESC
'''

tap.search(pregunta_conteo).to_table()

## 5. Separar por instrumento y por fecha

Se filtra la tabla con **máscaras** (arreglos de True/False): `obs['instrument_name'] == 'HARPS'` da True en las filas de HARPS. `&` significa "y", y cada condición va entre paréntesis.

In [ ]:
HARPS = obs[(obs['instrument_name'] == 'HARPS') & (obs['dataproduct_type'] == 'spectrum')]
ESPRESSO = obs[(obs['instrument_name'] == 'ESPRESSO') & (obs['dataproduct_type'] == 'spectrum')]
HARPS.sort('t_min')       # ordenar por fecha
ESPRESSO.sort('t_min')
print(f'Espectros de HARPS: {len(HARPS)}')
print(f'Espectros de ESPRESSO: {len(ESPRESSO)}')

`t_min` está en **MJD** (días desde 1858-11-17). `Time(..., format='mjd').iso` lo pasa a una fecha legible.

In [ ]:
fechas = Time(ESPRESSO['t_min'], format='mjd')
ESPRESSO['fechas'] = fechas.iso          # nueva columna con la fecha legible
ESPRESSO['dp_id', 'fechas', 'snr'][:10]

### Corte de ESPRESSO en junio de 2019

En junio de 2019 ESPRESSO tuvo una intervención técnica que introduce un **offset** en las velocidades radiales. Los datos de antes y de después se comportan como si fueran de dos instrumentos distintos.

**Decisión del equipo (con Camilo):** para ESPRESSO usamos **solo los datos de después de junio de 2019**.

Para comparar, hay que poner las dos cosas en el mismo formato: se pasa la fecha de corte a MJD con `Time('2019-06-01').mjd`.

In [ ]:
corte = Time('2019-06-01').mjd
antes = ESPRESSO[ESPRESSO['t_min'] < corte]
despues = ESPRESSO[ESPRESSO['t_min'] >= corte]
print('Antes de junio de 2019:', len(antes), '| Después:', len(despues))

## 6. Archivos de un producto

Un producto es un paquete con varios archivos. Para ver qué trae se consulta **otra tabla** del mismo servicio: `phase3v2.product_files`, que tiene **una fila por archivo**.

**¿De dónde salió esta tabla?** Del notebook oficial de ESO *"How to search for ancillary files"*, en https://archive.eso.org/programmatic.

| Columna | Qué guarda |
|---|---|
| `product_id` | A qué producto pertenece (= `dp_id` de ObsCore). |
| `archive_id` | Identificador del archivo. |
| `eso_category` | Qué es: empieza por `SCIENCE.` o `ANCILLARY.`. |
| `original_filename` | Nombre original del archivo. Suele decir mucho de qué es. |
| `access_url` | Enlace de descarga directa. |
| `access_estsize` | Tamaño aproximado, en KB. |

**Resultado para un producto de ESPRESSO:** 6 archivos: `SCIENCE.SPECTRUM`, `ANCILLARY.SPECTRUM`, `ANCILLARY.CCF`, `ANCILLARY.DRIFT`, **`ANCILLARY.2DECHELLE.TAR`** (~137 MB, donde debería estar el S2D) y `ANCILLARY.PREVIEW`.

In [ ]:
un_producto = ESPRESSO['dp_id'][0]
print(f'Producto: {un_producto}')

archivos = tap.search(f'''
SELECT archive_id, eso_category, original_filename, access_url, access_estsize
FROM phase3v2.product_files
WHERE product_id = '{un_producto}'
''').to_table()
archivos['archive_id', 'eso_category', 'original_filename', 'access_estsize']

## 7. Descargar

Función reutilizable. Sus decisiones de diseño:
- **Si el archivo ya existe, no lo vuelve a bajar.** Si la descarga se cae a la mitad, se vuelve a correr y sigue desde donde iba.
- **`stream=True` + `iter_content`**: baja de a 1 MB, sin cargar el archivo entero en la RAM.
- **`raise_for_status()`**: si el servidor responde con error (404, 403…), se detiene en vez de guardar la página de error como si fuera un FITS.
- **Escribe en `.part` y renombra al final**: un archivo que termina en `.fits` siempre está completo.
- `parents=True` crea las carpetas intermedias, y `exist_ok=True` evita el error si la carpeta ya existe.
- `'wb'` = escribir en binario (los FITS son bytes, no texto).

In [ ]:
def descargar(url, destino):
    destino = Path(destino)
    if destino.exists():
        print(f'Ya existe, no lo bajo otra vez: {destino.name}')
        return destino
    destino.parent.mkdir(parents=True, exist_ok=True)
    temporal = destino.with_name(destino.name + '.part')

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(temporal, 'wb') as f:
            for pedazo in r.iter_content(chunk_size=1024*1024):   # 1 MB
                f.write(pedazo)

    temporal.rename(destino)
    print(f'Descargado: {destino.name}')
    return destino

Prueba con el archivo de **ciencia** (el espectro 1D) de un producto de ESPRESSO. **Este no es el que usa LBL**; sirve para revisar que la descarga funciona.

En Colab, las rutas relativas quedan dentro de `/content/` (panel de archivos a la izquierda). **Se borran al cerrar la sesión.**

In [ ]:
CARPETA = Path('datos_LHS_1140_SCIENCE')

es_ciencia = [str(c).startswith('SCIENCE') for c in archivos['eso_category']]
archivo_ciencia = archivos[es_ciencia][0]
print(archivo_ciencia['archive_id'], archivo_ciencia['eso_category'])

ruta = descargar(archivo_ciencia['access_url'],
                 CARPETA / 'ESPRESSO' / f"{archivo_ciencia['archive_id']}.fits")

**Prueba de que no repite descargas:** al correr la misma celda otra vez, debe decir "Ya existe".

In [ ]:
ruta = descargar(archivo_ciencia['access_url'],
                 CARPETA / 'ESPRESSO' / f"{archivo_ciencia['archive_id']}.fits")

## 8. Abrir el archivo y revisarlo

Un **FITS** es una lista de bloques (HDU). Cada uno tiene un **header** (metadatos) y, a veces, **datos** (una imagen o una tabla).
- `hdul.info()`: resumen de los bloques.
- `hdul[0].header`: metadatos; se leen como diccionario. `h.get('CLAVE')` devuelve `None` si la clave no existe.

**Resultado:** el HDU 0 solo tiene header (1674 líneas) y el HDU 1 es una tabla de **1 fila × 12 columnas**, donde cada celda es un arreglo de 443 262 puntos: el espectro **1D** con todos los órdenes pegados. Fecha 2018-10-25 (antes del corte de 2019).

In [ ]:
with fits.open(ruta) as hdul:
    hdul.info()
    h = hdul[0].header
    print()
    print(f"Instrumento: {h.get('INSTRUME')}")
    print(f"Objeto: {h.get('OBJECT')}")
    print(f"Fecha: {h.get('DATE-OBS')}")
    print(f"MJD: {h.get('MJD-OBS')}")

Nombres y unidades de las 12 columnas (`D` = decimal float64, `J` = entero, normalmente banderas de calidad):

In [ ]:
with fits.open(ruta) as hdul:
    print(hdul[1].columns)

In [ ]:
with fits.open(ruta) as hdul:
    tabla = hdul[1].data
    nombres = tabla.columns.names
    unidad_onda = hdul[1].columns['WAVE'].unit

    # La columna de flujo puede llamarse FLUX o FLUX_REDUCED según el producto
    col_flujo = 'FLUX' if 'FLUX' in nombres else [n for n in nombres if n.startswith('FLUX')][0]

    onda = tabla['WAVE'][0]      # [0] = la única fila; cada celda es el arreglo completo
    flujo = tabla[col_flujo][0]

print(f'Columna de flujo usada: {col_flujo}')
print(f'Unidad de onda: {unidad_onda}')
print(f'Rango: {onda.min()} - {onda.max()}')

Gráfica de control alrededor de Ca II K (3933.66 Å), solo para ver que los datos tienen sentido.

In [ ]:
linea = 393.366 if 'nm' in str(unidad_onda) else 3933.66
ventana = 0.5 if 'nm' in str(unidad_onda) else 5

zona = (onda > linea - ventana) & (onda < linea + ventana)

plt.figure(figsize=(8, 4))
plt.plot(onda[zona], flujo[zona], lw=0.8)
plt.axvline(linea, color='gray', ls='--', label='Ca II K')
plt.xlabel(f'Longitud de onda [{unidad_onda}]')
plt.ylabel('Flujo')
plt.title('LHS 1140, ESPRESSO')
plt.legend()
plt.show()

## 9. Varios productos y manifiesto

El mismo procedimiento en un bucle (solo 3 productos, como prueba). Al final se guarda un **manifiesto** CSV con qué se bajó, de qué producto, de qué fecha y dónde quedó.

In [ ]:
registro = []

for prod in ESPRESSO[:3]:
    lista = tap.search(f'''
        SELECT archive_id, eso_category, access_url
        FROM phase3v2.product_files
        WHERE product_id = '{prod["dp_id"]}'
    ''').to_table()

    for arch in lista:
        if str(arch['eso_category']).startswith('SCIENCE'):
            r = descargar(arch['access_url'], CARPETA / 'ESPRESSO' / f"{arch['archive_id']}.fits")
            registro.append({'dp_id': prod['dp_id'], 'fecha': prod['fechas'],
                             'archive_id': arch['archive_id'], 'ruta': str(r)})

print(len(registro), 'archivos en el registro')

In [ ]:
manifiesto = Table(rows=registro)
manifiesto.write(CARPETA / 'manifiesto.csv', format='ascii.csv', overwrite=True)
manifiesto

---
# 10. Punto de partida para HARPS

## 10.1 Archivos de un producto de HARPS

Misma consulta de la sección 6, con un producto de HARPS.

**Resultado:** un producto de HARPS tiene solo **2 archivos**:

| `eso_category` | `original_filename` | Tamaño |
|---|---|---|
| `SCIENCE.SPECTRUM` | `HARPS.2015-11-23T02:59:06.381_s1d_A_DRS_HARPS_3.8_ESOSDP.fits` | ~5 MB |
| `ANCILLARY.HARPSTAR` | `HARPS.2015-11-23T02:59:06.381_DRS_HARPS_3.8.tar` | ~7 MB |

`DRS_HARPS_3.8` indica que fue reducido con el **pipeline original de HARPS** (versión 3.8). En la terminología de LBL, esto es formato **ORIG**: el espectro por órdenes **no** viene corregido por blaze y LBL necesita el archivo de blaze aparte.

In [ ]:
un_producto_harps = HARPS['dp_id'][0]
print(f'Producto: {un_producto_harps}')

archivos_harps = tap.search(f'''
SELECT archive_id, eso_category, original_filename, access_url, access_estsize
FROM phase3v2.product_files
WHERE product_id = '{un_producto_harps}'
''').to_table()

archivos_harps['eso_category', 'original_filename', 'access_estsize'].pprint_all()

## 10.2 Abrir el `.tar`

**Librería `tarfile`** (viene con Python). Un `.tar` es una "caja" que junta varios archivos en uno.
- `tarfile.open(ruta)`: abre la caja (sin sacar nada).
- `.getmembers()`: lista lo que hay adentro (`.name`, `.size`).
- `.extractfile(miembro)`: lee **un** archivo de adentro sin guardarlo en disco.

Primero se baja el `.tar` (~7 MB) con la misma función `descargar`.

In [ ]:
es_tar = [str(c) == 'ANCILLARY.HARPSTAR' for c in archivos_harps['eso_category']]
fila_tar = archivos_harps[es_tar][0]

ruta_tar = descargar(fila_tar['access_url'],
                     Path('datos_LHS_1140_TAR') / 'HARPS' / f"{fila_tar['archive_id']}.tar")

with tarfile.open(ruta_tar) as caja:
    for miembro in caja.getmembers():
        print(f'{miembro.size/1e6:8.2f} MB   {miembro.name}')

**Resultado:** 11 archivos de la misma observación. Los importantes:

| Archivo | Qué es | ¿Lo usa LBL? |
|---|---|---|
| `..._e2ds_A.fits` | Espectro **por órdenes**, fibra A (la estrella). Imagen de **4096 píxeles × 72 órdenes**. | **Sí** |
| `..._e2ds_B.fits` | Lo mismo, fibra B (calibración/cielo). | No |
| `..._s1d_A/B.fits` | Espectros 1D (todos los órdenes pegados). | No |
| `..._ccf_*`, `..._bis_*` | Correlación cruzada y bisector del pipeline. | No (sirven para comparar) |
| `..._INT_GUIDE.fits` | Imagen de guiado. | No |

**El blaze no viene en el `.tar`.**

## 10.3 ¿Qué blaze usó cada espectro?

El pipeline anota en el **header** de cada `e2ds` qué archivos de calibración usó. Se puede leer sin extraer el archivo al disco:
- `caja.extractfile(miembro).read()` lee los bytes del archivo dentro del `.tar`.
- `io.BytesIO(bytes)` envuelve esos bytes para que `fits.open` los trate como si fueran un archivo.

In [ ]:
with tarfile.open(ruta_tar) as caja:
    m = [x for x in caja.getmembers() if x.name.endswith('e2ds_A.fits')][0]
    datos = caja.extractfile(m).read()

with fits.open(io.BytesIO(datos)) as hdul:
    hdul.info()
    h = hdul[0].header
    # Buscar cualquier clave del header cuyo nombre mencione BLAZE, FLAT o el archivo de onda
    for clave in h:
        if any(s in clave.upper() for s in ['BLAZE', 'FLAT FILE', 'TH FILE']):
            print(clave, '=', h[clave])

**Resultado** (producto del 2015-11-23):
```
ESO DRS BLAZE FILE   = HARPS.2015-11-22T20:22:37.879_blaze_A.fits
ESO DRS CAL FLAT FILE = HARPS.2015-11-22T20:22:37.879_flat_A.fits
ESO DRS CAL TH FILE   = HARPS.2015-11-22T20:37:37.636_wave_A.fits
```
El blaze es **uno por calibración** (de la tarde de esa noche), no uno por estrella ni por producto. Varios espectros de noches cercanas pueden compartir el mismo.

## 10.4 Decisiones y pendientes para HARPS

- **Blaze:** Camilo nos pasará el blaze que él usó. **Está construido de una forma particular, así que hay que documentarlo en el informe** (cómo se hizo y por qué).
- **Siguiente paso sugerido:**
  1. Para los 296 productos, bajar el `.tar` (~7 MB × 296 ≈ 2 GB).
  2. Sacar solo el `e2ds_A` de cada uno.
  3. Leer en cada header el `ESO DRS BLAZE FILE` y armar la lista de **blazes distintos** que se necesitan. Esto sirve para comparar con lo que entregue Camilo.
  4. Organizar los archivos en la estructura de LBL: `science/LHS1140/` y `calib/`.
- **Pregunta abierta:** ¿LBL necesita también el archivo `wave_A` (solución de longitud de onda) que aparece en el header, o la lee del propio `e2ds`? Revisar en la documentación de LBL (formato HARPS ORIG).